# CATH backbone dataset audit for the V4/V4i diffusion model

This notebook audits the CATH backbone dataset used by the V4/V4i protein backbone diffusion notebooks. It is designed to be run without GPU training and to produce report-ready tables and plots.

The first part recreates the same data path used before training in V4/V4i:

- preserve the provided `train` / `validation` / `test` split;
- treat `cath_nodes` as auxiliary metadata rather than a training split;
- extract the four backbone atoms `N`, `CA`, `C`, and `O`;
- create a residue mask for positions with complete finite backbone coordinates;
- pad/crop examples to a 256-residue window;
- run the same style of per-record audit used before model training;
- filter only empty or invalid examples;
- expand long training chains into overlapping windows;
- compute train-only coordinate normalisation statistics after centring.

The second part adds the extra dataset characterisation requested for the report: length distributions, chunking bias, geometry distributions, radius of gyration, CATH-label composition, and split-integrity checks.

In [5]:
import json
import math
import os
import random
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Iterator

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MAX_SEQ_LENGTH = int(os.environ.get("CATH_AUDIT_MAX_SEQ_LENGTH", "256"))
TRAIN_CHUNK_STRIDE = int(os.environ.get("CATH_AUDIT_TRAIN_CHUNK_STRIDE", "128"))
BACKBONE_ATOMS = ("N", "CA", "C", "O")
SHOW_FIGURES = os.environ.get("CATH_AUDIT_SHOW_FIGURES", "0").strip().lower() in {"1", "true", "yes", "y", "on"}

# The notebook prefers explicit environment variables, but also works in this sandbox
# and when placed next to the two CATH data files.
NOTEBOOK_DIR = Path.cwd()
DEFAULT_DATA_CANDIDATES = [
    Path(os.environ.get("CATH_DATA_DIR", "")) if os.environ.get("CATH_DATA_DIR") else None,
    NOTEBOOK_DIR,
    Path("/mnt/data"),
]

def resolve_data_dir() -> Path:
    for candidate in DEFAULT_DATA_CANDIDATES:
        if candidate is None:
            continue
        candidate = candidate.expanduser().resolve()
        if (candidate / "chain_set.jsonl").exists() and (candidate / "chain_set_splits.json").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find chain_set.jsonl and chain_set_splits.json. "
        "Set CATH_DATA_DIR to the directory containing the files."
    )

DATA_DIR = Path("../data/raw").resolve()
CHAIN_SET_PATH = DATA_DIR / "chain_set.jsonl"
SPLITS_PATH = DATA_DIR / "chain_set_splits.json"

OUTPUT_DIR = Path(os.environ.get("CATH_DATA_AUDIT_OUTPUT_DIR", str(DATA_DIR / "outputs" / "data_audit")))
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"chain_set.jsonl: {CHAIN_SET_PATH}")
print(f"chain_set_splits.json: {SPLITS_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")
print(f"TRAIN_CHUNK_STRIDE: {TRAIN_CHUNK_STRIDE}")
print(f"SHOW_FIGURES: {SHOW_FIGURES}")

Data directory: /Users/mitsenkov/PycharmProjects/latent-structure-diffusion/data/raw
chain_set.jsonl: /Users/mitsenkov/PycharmProjects/latent-structure-diffusion/data/raw/chain_set.jsonl
chain_set_splits.json: /Users/mitsenkov/PycharmProjects/latent-structure-diffusion/data/raw/chain_set_splits.json
Output directory: /Users/mitsenkov/PycharmProjects/latent-structure-diffusion/data/raw/outputs/data_audit
MAX_SEQ_LENGTH: 256
TRAIN_CHUNK_STRIDE: 128
SHOW_FIGURES: False


## V4/V4i-compatible helper functions

The project package is not required here. The helper functions below are self-contained equivalents of the V4/V4i preprocessing path, using the same backbone atom order, residue masking, fixed-length windowing, centring, and train-only normalisation logic.

A small correction is used for adjacent-residue geometry: distances such as adjacent Cα and peptide C–N are calculated only for neighbouring residue indices where **both** residues are valid, rather than compressing valid residues together and accidentally creating artificial neighbours across gaps.

In [6]:
def canonical_split_name(name: str) -> str:
    return {"val": "validation", "validation": "validation", "train": "train", "test": "test"}.get(str(name), str(name))


def load_split_data(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def build_split_lookup(split_data: dict) -> dict[str, str]:
    split_lookup: dict[str, str] = {}

    # Preserve the true train/validation/test buckets first.
    for column in ("train", "validation", "test"):
        for record_name in split_data.get(column, []):
            split_lookup.setdefault(record_name, canonical_split_name(column))

    # cath_nodes is auxiliary metadata; use it only for records not already assigned.
    for record_name in split_data.get("cath_nodes", {}).keys():
        split_lookup.setdefault(record_name, "cath_nodes")

    return split_lookup


def iter_chain_records(path: Path) -> Iterator[dict]:
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)


def extract_backbone_from_coords_dict(coords_dict: dict) -> tuple[np.ndarray, np.ndarray]:
    """Return (coords, residue_mask) with shape (L, 4, 3) and mask shape (L,).

    This mirrors the V4/V4i assumption that a residue is usable only when all four
    backbone atoms are present and finite. Invalid/padded residues are zeroed and
    excluded by the mask.
    """
    arrays = [np.asarray(coords_dict.get(atom, []), dtype=np.float32) for atom in BACKBONE_ATOMS]
    lengths = [array.shape[0] for array in arrays if array.ndim == 2 and array.shape[-1] == 3]
    n_residues = max(lengths, default=0)

    if n_residues == 0:
        return np.zeros((0, len(BACKBONE_ATOMS), 3), dtype=np.float32), np.zeros(0, dtype=bool)

    coords = np.zeros((n_residues, len(BACKBONE_ATOMS), 3), dtype=np.float32)
    residue_mask = np.ones(n_residues, dtype=bool)

    for atom_idx, array in enumerate(arrays):
        atom_valid = np.zeros(n_residues, dtype=bool)
        if array.ndim == 2 and array.shape[-1] == 3:
            usable = min(n_residues, array.shape[0])
            finite = np.isfinite(array[:usable]).all(axis=-1)
            coords[:usable, atom_idx, :] = np.where(finite[:, None], array[:usable], 0.0)
            atom_valid[:usable] = finite
        residue_mask &= atom_valid

    coords[~residue_mask] = 0.0
    return coords, residue_mask


def pad_or_crop_backbone(coords: np.ndarray, mask: np.ndarray, max_length: int) -> tuple[np.ndarray, np.ndarray, bool]:
    n_residues = int(coords.shape[0])
    out_coords = np.zeros((max_length, len(BACKBONE_ATOMS), 3), dtype=np.float32)
    out_mask = np.zeros(max_length, dtype=bool)
    kept = min(n_residues, max_length)
    if kept > 0:
        out_coords[:kept] = coords[:kept]
        out_mask[:kept] = mask[:kept]
    return out_coords, out_mask, n_residues > max_length


def centre_coordinates(coords: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Centre a fixed-window backbone by the mean valid backbone-atom position."""
    centred = coords.copy()
    valid = mask.astype(bool)
    if valid.any():
        centre = centred[valid].reshape(-1, 3).mean(axis=0)
        centred[valid] -= centre.astype(np.float32)
    return centred


def radius_of_gyration(coords: np.ndarray, mask: np.ndarray, atom_index: int = 1) -> float:
    valid = mask.astype(bool)
    if valid.sum() == 0:
        return float("nan")
    atom_coords = coords[valid, atom_index, :]
    centre = atom_coords.mean(axis=0)
    return float(np.sqrt(((atom_coords - centre) ** 2).sum(axis=-1).mean()))


def backbone_structure_summary(coords: np.ndarray, mask: np.ndarray) -> dict[str, float | int]:
    """V4/V4i-style structural summary for one fixed-size backbone window."""
    valid = mask.astype(bool)
    n_valid = int(valid.sum())
    summary = {
        "n_residues": n_valid,
        "mean_adjacent_ca": np.nan,
        "fraction_adjacent_ca_in_band": np.nan,
        "mean_n_ca": np.nan,
        "mean_ca_c": np.nan,
        "mean_c_o": np.nan,
        "mean_c_n": np.nan,
        "radius_of_gyration": np.nan,
    }
    if n_valid == 0:
        return summary

    n = coords[:, 0, :]
    ca = coords[:, 1, :]
    c = coords[:, 2, :]
    o = coords[:, 3, :]

    summary["mean_n_ca"] = float(np.linalg.norm(ca[valid] - n[valid], axis=-1).mean())
    summary["mean_ca_c"] = float(np.linalg.norm(c[valid] - ca[valid], axis=-1).mean())
    summary["mean_c_o"] = float(np.linalg.norm(o[valid] - c[valid], axis=-1).mean())
    summary["radius_of_gyration"] = radius_of_gyration(coords, valid, atom_index=1)

    if len(valid) > 1:
        valid_pair = valid[:-1] & valid[1:]
        if valid_pair.any():
            adjacent_ca = np.linalg.norm(ca[1:] - ca[:-1], axis=-1)[valid_pair]
            c_n = np.linalg.norm(n[1:] - c[:-1], axis=-1)[valid_pair]
            summary["mean_adjacent_ca"] = float(adjacent_ca.mean())
            summary["fraction_adjacent_ca_in_band"] = float(((adjacent_ca >= 3.6) & (adjacent_ca <= 4.0)).mean())
            summary["mean_c_n"] = float(c_n.mean())
    return summary


def audit_record(row: dict, split_name: str, max_length: int) -> dict:
    record_id = row.get("name")
    try:
        coords, residue_mask = extract_backbone_from_coords_dict(row.get("coords", {}))
        raw_length = int(coords.shape[0])
        real_length = int(residue_mask.sum())
        coords_fixed, mask_fixed, truncated = pad_or_crop_backbone(coords, residue_mask, max_length)
        has_nan = bool(np.isnan(coords_fixed).any())
        has_inf = bool(np.isinf(coords_fixed).any())
        zero_real_residues = int(((np.abs(coords_fixed).sum(axis=(1, 2)) == 0.0) & mask_fixed).sum())
        summary = backbone_structure_summary(coords_fixed, mask_fixed)
        return {
            "record_id": record_id,
            "split": split_name,
            "raw_length": raw_length,
            "real_length": real_length,
            "padded_length": int(mask_fixed.shape[0]),
            "model_valid_length": int(mask_fixed.sum()),
            "truncated": bool(truncated),
            "has_nan": has_nan,
            "has_inf": has_inf,
            "zero_real_residues": zero_real_residues,
            "keep_example": bool(real_length > 0 and not has_nan and not has_inf),
            **summary,
        }
    except Exception as exc:
        return {
            "record_id": record_id,
            "split": split_name,
            "raw_length": np.nan,
            "real_length": 0,
            "padded_length": max_length,
            "model_valid_length": 0,
            "truncated": False,
            "has_nan": True,
            "has_inf": True,
            "zero_real_residues": 0,
            "keep_example": False,
            "audit_error": str(exc),
            "n_residues": 0,
            "mean_adjacent_ca": np.nan,
            "fraction_adjacent_ca_in_band": np.nan,
            "mean_n_ca": np.nan,
            "mean_ca_c": np.nan,
            "mean_c_o": np.nan,
            "mean_c_n": np.nan,
            "radius_of_gyration": np.nan,
        }


def chunk_starts_for_length(raw_length: int, max_length: int, stride: int) -> list[int]:
    if raw_length <= max_length:
        return [0]
    starts = list(range(0, raw_length - max_length + 1, stride))
    final_start = raw_length - max_length
    if starts[-1] != final_start:
        starts.append(final_start)
    return starts


def save_table(frame: pd.DataFrame, name: str) -> Path:
    path = TABLE_DIR / f"{name}.csv"
    frame.to_csv(path, index=False)
    return path


def save_current_figure(name: str) -> Path:
    path = FIGURE_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    return path


@dataclass
class RunningStats:
    count: int = 0
    total: float = 0.0
    total_sq: float = 0.0

    def update(self, values: np.ndarray) -> None:
        values = np.asarray(values, dtype=np.float64).ravel()
        values = values[np.isfinite(values)]
        if values.size == 0:
            return
        self.count += int(values.size)
        self.total += float(values.sum())
        self.total_sq += float((values ** 2).sum())

    @property
    def mean(self) -> float:
        return self.total / max(self.count, 1)

    @property
    def variance(self) -> float:
        if self.count <= 1:
            return 0.0
        return max((self.total_sq - (self.total ** 2) / self.count) / (self.count - 1), 0.0)

    @property
    def std(self) -> float:
        return float(np.sqrt(self.variance))


class ReservoirSampler:
    """A lightweight capped sampler for plotting large geometry distributions.

    It keeps the notebook fast by storing at most `max_size` values per metric. The
    exact mean/std above are computed over all distances; percentiles/histograms use
    this bounded sample.
    """
    def __init__(self, max_size: int, seed: int = SEED):
        self.max_size = max_size
        self.values: list[float] = []
        self.rng = np.random.default_rng(seed)

    def update(self, values: np.ndarray) -> None:
        values = np.asarray(values, dtype=np.float64).ravel()
        values = values[np.isfinite(values)]
        if values.size == 0 or len(self.values) >= self.max_size:
            return
        remaining = self.max_size - len(self.values)
        if values.size > remaining:
            idx = self.rng.choice(values.size, size=remaining, replace=False)
            values = values[idx]
        self.values.extend(values.tolist())

    def array(self) -> np.ndarray:
        return np.asarray(self.values, dtype=np.float64)


## 1. Load split assignments and run the V4/V4i-style audit

This cell preserves the true `train`, `validation`, and `test` buckets first. The `cath_nodes` mapping is kept as auxiliary metadata and only used to label records that are not already assigned to one of the modelling splits.

The audit is intentionally similar to the pre-training audit in V4/V4i: it reports raw length, valid-residue length, fixed-window length, truncation, invalid values, the final keep/drop decision, local backbone geometry, and Cα radius of gyration.

In [7]:
split_data = load_split_data(SPLITS_PATH)
split_lookup = build_split_lookup(split_data)

split_counts = Counter()
metadata_rows = []
audit_rows = []
seen_names = set()

for record_index, row in enumerate(iter_chain_records(CHAIN_SET_PATH), start=1):
    if record_index % 5000 == 0:
        print(f"Processed {record_index:,} records during audit...")
    name = row.get("name")
    seen_names.add(name)
    split = canonical_split_name(split_lookup.get(name, "unknown"))
    split_counts[split] += 1

    cath_labels = row.get("CATH") or []
    if not isinstance(cath_labels, list):
        cath_labels = [cath_labels]
    metadata_rows.append({
        "record_id": name,
        "split": split,
        "seq_length": len(row.get("seq", "")),
        "num_chains": row.get("num_chains", np.nan),
        "n_cath_labels": len(cath_labels),
        "cath_labels": ";".join(map(str, cath_labels)),
    })

    if split in {"train", "validation", "test"}:
        audit_rows.append(audit_record(row, split, MAX_SEQ_LENGTH))

print("Audit scan complete; building audit dataframes...")
metadata_df = pd.DataFrame(metadata_rows)
audit_df = pd.DataFrame(audit_rows)
print("Audit dataframes built.")

raw_split_summary = (
    metadata_df.groupby("split", as_index=False)
    .agg(raw_records=("record_id", "count"), median_seq_length=("seq_length", "median"), mean_seq_length=("seq_length", "mean"))
    .sort_values("split")
)

filter_summary = (
    audit_df.groupby("split", as_index=False)
    .agg(
        total_examples=("record_id", "count"),
        kept_examples=("keep_example", "sum"),
        invalid_examples=("keep_example", lambda s: int((~s.astype(bool)).sum())),
        truncated_examples=("truncated", "sum"),
        median_real_length=("real_length", "median"),
        mean_real_length=("real_length", "mean"),
        max_real_length=("real_length", "max"),
    )
)
filter_summary["filtered_out"] = filter_summary["total_examples"] - filter_summary["kept_examples"]
filter_summary["truncated_fraction"] = filter_summary["truncated_examples"] / filter_summary["total_examples"]
filter_summary["filtered_fraction"] = filter_summary["filtered_out"] / filter_summary["total_examples"]

kept_audit = audit_df[audit_df["keep_example"].astype(bool)].copy()
kept_audit["padding_residues_in_model_window"] = MAX_SEQ_LENGTH - kept_audit["model_valid_length"]
kept_audit["padding_fraction_in_model_window"] = kept_audit["padding_residues_in_model_window"] / MAX_SEQ_LENGTH
padding_summary = (
    kept_audit.groupby("split", as_index=False)
    .agg(
        kept_examples=("record_id", "count"),
        mean_model_valid_length=("model_valid_length", "mean"),
        median_model_valid_length=("model_valid_length", "median"),
        mean_padding_fraction_in_model_window=("padding_fraction_in_model_window", "mean"),
        truncated_fraction=("truncated", "mean"),
    )
)

normalisation_summary = pd.DataFrame([{
    "source": "documented_from_v4_v4i_training_notebook",
    "training_notebook_helper": "build_normalization_stats_from_dataframe(train_df)",
    "centering_helper": "centre_coordinates(coords, mask)",
    "statistics_split": "filtered training parent chains only",
    "validation_or_test_used": False,
    "note": "The standalone audit avoids a second heavy coordinate pass; the submitted V4/V4i training notebook computes the actual mean/std values before training.",
}])

print("Saving audit tables...")
save_table(metadata_df.drop(columns=["cath_labels"]), "metadata_manifest_without_coords")
save_table(raw_split_summary, "raw_split_summary")
save_table(audit_df, "v4_style_audit_table")
save_table(filter_summary, "filter_summary")
save_table(padding_summary, "padding_summary")
save_table(normalisation_summary, "documented_train_only_coordinate_normalisation")
print("Audit tables saved.")

print("Raw split summary, including auxiliary/non-model records:")
display(raw_split_summary)
print("V4/V4i-style filtering summary for modelling splits:")
display(filter_summary)
print("Padding/truncation summary for the fixed 256-residue model window:")
display(padding_summary)
print("Train-only coordinate normalisation documentation:")
display(normalisation_summary)
print("Audit table preview:")
display(audit_df.head())

Processed 5,000 records during audit...
Processed 10,000 records during audit...
Processed 15,000 records during audit...
Processed 20,000 records during audit...
Audit scan complete; building audit dataframes...
Audit dataframes built.
Saving audit tables...
Audit tables saved.
Raw split summary, including auxiliary/non-model records:


,split,raw_records,median_seq_length,mean_seq_length
0,cath_nodes,1820,269.5,276.598901
1,test,1120,149.0,174.670536
2,train,18024,220.0,233.612850
3,unknown,96,218.0,221.677083
4,validation,608,158.0,187.208882


V4/V4i-style filtering summary for modelling splits:


,split,total_examples,kept_examples,invalid_examples,truncated_examples,median_real_length,mean_real_length,max_real_length,filtered_out,truncated_fraction,filtered_fraction
0,test,1120,1120,0,188,138.0,162.196429,497,0,0.167857,0.0
1,train,18024,18024,0,7424,204.0,218.666611,500,0,0.411895,0.0
2,validation,608,608,0,132,146.0,174.185855,475,0,0.217105,0.0


Padding/truncation summary for the fixed 256-residue model window:


,split,kept_examples,mean_model_valid_length,median_model_valid_length,mean_padding_fraction_in_model_window,truncated_fraction
0,test,1120,151.135714,137.0,0.409626,0.167857
1,train,18024,182.260652,200.0,0.288044,0.411895
2,validation,608,156.393092,146.0,0.389089,0.217105


Train-only coordinate normalisation documentation:


,source,training_notebook_helper,centering_helper,statistics_split,validation_or_test_used,note
0,documented_from_v4_v4i_training_notebook,build_normalization_stats_from_dataframe(train...,"centre_coordinates(coords, mask)",filtered training parent chains only,False,The standalone audit avoids a second heavy coo...


Audit table preview:


,record_id,split,raw_length,real_length,padded_length,model_valid_length,truncated,has_nan,has_inf,zero_real_residues,keep_example,n_residues,mean_adjacent_ca,fraction_adjacent_ca_in_band,mean_n_ca,mean_ca_c,mean_c_o,mean_c_n,radius_of_gyration
0,12as.A,train,330,327,256,253,True,False,False,0,True,253,3.801517,1.000000,1.458675,1.524882,1.231658,1.330370,18.822926
1,132l.A,train,129,129,256,129,False,False,False,0,True,129,3.790657,1.000000,1.451698,1.522182,1.229839,1.329097,13.762529
2,153l.A,train,185,185,256,185,False,False,False,0,True,185,3.793202,1.000000,1.452797,1.520918,1.227060,1.328758,14.764102
3,16pk.A,train,415,415,256,256,True,False,False,0,True,256,3.802165,0.996078,1.458607,1.525249,1.233423,1.329715,21.210670
4,16vp.A,train,366,311,256,256,True,False,False,0,True,256,3.808722,1.000000,1.458273,1.526411,1.231872,1.330091,22.820541


In [8]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_df = raw_split_summary.sort_values("raw_records", ascending=False)
ax.bar(plot_df["split"], plot_df["raw_records"])
ax.set_title("Records by assigned split")
ax.set_ylabel("Number of records")
ax.set_xlabel("Split")
plt.xticks(rotation=30, ha="right")
save_current_figure("split_record_counts")
plt.show() if SHOW_FIGURES else plt.close(fig)

## 2. Length distribution and the 256-residue fixed window

The model uses a fixed 256-residue input window. This section quantifies how much of the dataset is shorter than, equal to, or longer than that window, and therefore motivates mask-aware padding/cropping and overlapping training chunks.

In [9]:
length_rows = []
for split, group in kept_audit.groupby("split"):
    lengths = group["real_length"].astype(float).to_numpy()
    length_rows.append({
        "split": split,
        "n": int(len(lengths)),
        "min": float(np.min(lengths)),
        "p05": float(np.percentile(lengths, 5)),
        "p25": float(np.percentile(lengths, 25)),
        "median": float(np.median(lengths)),
        "mean": float(np.mean(lengths)),
        "p75": float(np.percentile(lengths, 75)),
        "p95": float(np.percentile(lengths, 95)),
        "max": float(np.max(lengths)),
        "fraction_shorter_than_window": float((lengths < MAX_SEQ_LENGTH).mean()),
        "fraction_equal_to_window": float((lengths == MAX_SEQ_LENGTH).mean()),
        "fraction_longer_than_window": float((lengths > MAX_SEQ_LENGTH).mean()),
    })
length_summary = pd.DataFrame(length_rows).sort_values("split")
save_table(length_summary, "length_distribution_summary")
display(length_summary)

,split,n,min,p05,p25,median,mean,p75,p95,max,fraction_shorter_than_window,fraction_equal_to_window,fraction_longer_than_window
0,test,1120,40.0,62.0,106.0,138.0,162.196429,206.0,326.0,497.0,0.866964,0.000893,0.132143
1,train,18024,39.0,70.0,124.0,204.0,218.666611,301.0,418.0,500.0,0.633489,0.002663,0.363848
2,validation,608,40.0,61.0,113.0,146.0,174.185855,217.0,362.3,475.0,0.807566,0.001645,0.190789


In [10]:
fig, ax = plt.subplots(figsize=(8, 5))
for split in ["train", "validation", "test"]:
    values = kept_audit.loc[kept_audit["split"] == split, "real_length"].astype(float).to_numpy()
    if len(values):
        ax.hist(values, bins=50, alpha=0.45, label=split)
ax.axvline(MAX_SEQ_LENGTH, linestyle="--", linewidth=2, label=f"model window = {MAX_SEQ_LENGTH}")
ax.set_title("Valid residue length distribution by split")
ax.set_xlabel("Valid residues per parent chain")
ax.set_ylabel("Number of records")
ax.legend()
save_current_figure("length_distribution_by_split")
plt.show() if SHOW_FIGURES else plt.close(fig)

## 3. Overlapping training chunks

V4/V4i filters parent chains first, then expands **training** chains longer than 256 residues into overlapping windows with stride 128. Validation and test records are not expanded into overlapping windows in the same way; they remain parent-chain records padded/cropped to the model window.

This section audits the exact chunk-count logic used by the training notebook and highlights the trade-off: chunking improves coverage of long chains, but longer parent chains contribute more training windows.

In [11]:
kept_names_by_split = {
    split: set(group.loc[group["keep_example"].astype(bool), "record_id"].tolist())
    for split, group in audit_df.groupby("split")
}
kept_train_names = kept_names_by_split.get("train", set())

# Chunk counts are determined entirely by the raw parent-chain length, max window, and stride.
# This mirrors the V4/V4i training-manifest logic without re-reading coordinates unnecessarily.
train_kept_parent_audit = audit_df[(audit_df["split"] == "train") & (audit_df["keep_example"].astype(bool))].copy()
chunk_rows = []
for _, parent in train_kept_parent_audit.iterrows():
    name = parent["record_id"]
    raw_length = int(parent["raw_length"])
    parent_valid_length = int(parent["real_length"])
    for chunk_number, start in enumerate(chunk_starts_for_length(raw_length, MAX_SEQ_LENGTH, TRAIN_CHUNK_STRIDE)):
        end = min(start + MAX_SEQ_LENGTH, raw_length)
        chunk_rows.append({
            "chunk_id": f"{name}__chunk_{chunk_number:03d}",
            "parent_chain_id": name,
            "chunk_number": chunk_number,
            "chunk_start": int(start),
            "chunk_end": int(end),
            "chunk_length": int(end - start),
            "parent_raw_length": raw_length,
            "parent_valid_length": parent_valid_length,
            "parent_was_longer_than_window": bool(raw_length > MAX_SEQ_LENGTH),
        })

chunk_df = pd.DataFrame(chunk_rows)
chunk_counts = (
    chunk_df.groupby("parent_chain_id", as_index=False)
    .agg(
        chunk_count=("chunk_id", "count"),
        parent_raw_length=("parent_raw_length", "first"),
        parent_valid_length=("parent_valid_length", "first"),
    )
    .sort_values("chunk_count", ascending=False)
)
chunk_summary = pd.DataFrame([{
    "train_parent_chains": int(len(chunk_counts)),
    "training_chunks": int(len(chunk_df)),
    "min_chunks_per_parent": int(chunk_counts["chunk_count"].min()),
    "median_chunks_per_parent": float(chunk_counts["chunk_count"].median()),
    "mean_chunks_per_parent": float(chunk_counts["chunk_count"].mean()),
    "max_chunks_per_parent": int(chunk_counts["chunk_count"].max()),
    "parents_with_multiple_chunks": int((chunk_counts["chunk_count"] > 1).sum()),
    "fraction_parents_with_multiple_chunks": float((chunk_counts["chunk_count"] > 1).mean()),
}])

save_table(chunk_df, "train_chunk_manifest_without_coords")
save_table(chunk_counts, "train_chunk_counts")
save_table(chunk_summary, "train_chunking_summary")

display(chunk_summary)
print("Most heavily represented parent chains after chunking:")
display(chunk_counts.head(10))

,train_parent_chains,training_chunks,min_chunks_per_parent,median_chunks_per_parent,mean_chunks_per_parent,max_chunks_per_parent,parents_with_multiple_chunks,fraction_parents_with_multiple_chunks
0,18024,27706,1,1.0,1.537173,3,7424,0.411895


Most heavily represented parent chains after chunking:


,parent_chain_id,chunk_count,parent_raw_length,parent_valid_length
13096,3oym.A,3,395,368
14589,4apm.A,3,437,339
14541,4ac9.C,3,482,471
14540,4aby.A,3,415,358
14539,4ab7.H,3,464,422
9894,2zai.A,3,497,472
2121,1l0q.A,3,391,391
9900,2zbk.A,3,389,347
9903,2zc0.A,3,407,405
14515,4a37.A,3,388,375


In [12]:
fig, ax = plt.subplots(figsize=(7, 4))
max_count = int(chunk_counts["chunk_count"].max())
bins = np.arange(0.5, max_count + 1.5, 1)
ax.hist(chunk_counts["chunk_count"], bins=bins)
ax.set_title("Training chunks per parent chain")
ax.set_xlabel("Number of overlapping chunks from one parent chain")
ax.set_ylabel("Number of parent chains")
ax.set_xticks(range(1, max_count + 1))
save_current_figure("train_chunks_per_parent_distribution")
plt.show() if SHOW_FIGURES else plt.close(fig)

## 4. Backbone geometry statistics from the V4/V4i audit

The V4/V4i audit already computes local backbone geometry for each model-facing parent window: N–CA, CA–C, C–O, peptide C–N, adjacent Cα distance, and the fraction of adjacent Cα distances in a 3.6–4.0 Å band. This section summarises those audited per-record means rather than running a second heavy all-coordinate pass.

In [22]:
geometry_metric_columns = {
    "N_CA": "mean_n_ca",
    "CA_C": "mean_ca_c",
    "C_O": "mean_c_o",
    "C_N": "mean_c_n",
    "adjacent_CA": "mean_adjacent_ca",
}

geometry_summary_rows = []
train_window_audit = kept_audit[kept_audit["split"] == "train"].copy()
for metric_name, column in geometry_metric_columns.items():
    values = train_window_audit[column].dropna().astype(float).to_numpy()
    geometry_summary_rows.append({
        "metric": metric_name,
        "n_parent_windows": int(len(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values, ddof=1)),
        "p05": float(np.percentile(values, 5)),
        "p25": float(np.percentile(values, 25)),
        "median": float(np.percentile(values, 50)),
        "p75": float(np.percentile(values, 75)),
        "p95": float(np.percentile(values, 95)),
    })

geometry_summary = pd.DataFrame(geometry_summary_rows)
save_table(geometry_summary, "training_parent_window_geometry_summary")
display(geometry_summary)

,metric,n_parent_windows,mean,std,p05,p25,median,p75,p95
0,N_CA,18024,1.460219,0.005775,1.454147,1.457545,1.459037,1.462114,1.467797
1,CA_C,18024,1.525520,0.003962,1.520259,1.523663,1.525134,1.527305,1.530937
2,C_O,18024,1.231636,0.003957,1.227519,1.230613,1.231478,1.233022,1.236629
3,C_N,18024,1.330052,0.006119,1.325536,1.328975,1.329842,1.331392,1.335170
4,adjacent_CA,18024,3.805123,0.014078,3.786814,3.798806,3.803693,3.810024,3.827333


In [14]:
print("Geometry summary table saved; per-metric geometry histograms are omitted in the lightweight run to keep the notebook fast. Use training_parent_window_geometry_summary.csv for report values.")

Geometry summary table saved; per-metric geometry histograms are omitted in the lightweight run to keep the notebook fast. Use training_parent_window_geometry_summary.csv for report values.


## 5. Radius of gyration and compactness

Radius of gyration is especially important because over-compactness/collapse was one of the main observed failure modes during sampling. The V4/V4i audit computes Cα radius of gyration for each model-facing parent window. This section summarises those values and also shows a chunk-weighted view, which reflects the fact that long parent chains contribute multiple training windows.

In [15]:
parent_rg_df = kept_audit[["record_id", "split", "real_length", "model_valid_length", "radius_of_gyration"]].copy()
parent_rg_df = parent_rg_df.rename(columns={"radius_of_gyration": "rg_ca_parent_model_window"})
save_table(parent_rg_df, "parent_model_window_radius_of_gyration")

chunk_weighted_rg_df = chunk_df.merge(
    parent_rg_df[parent_rg_df["split"] == "train"][["record_id", "rg_ca_parent_model_window"]],
    left_on="parent_chain_id",
    right_on="record_id",
    how="left",
).drop(columns=["record_id"])

rg_summary = pd.concat([
    parent_rg_df.groupby("split", as_index=False).agg(
        n=("record_id", "count"),
        median_model_valid_length=("model_valid_length", "median"),
        mean_rg=("rg_ca_parent_model_window", "mean"),
        median_rg=("rg_ca_parent_model_window", "median"),
        p05_rg=("rg_ca_parent_model_window", lambda s: float(np.percentile(s.dropna(), 5))),
        p95_rg=("rg_ca_parent_model_window", lambda s: float(np.percentile(s.dropna(), 95))),
    ).assign(dataset_view="parent_model_window"),
    pd.DataFrame([{
        "split": "train",
        "n": int(len(chunk_weighted_rg_df)),
        "median_model_valid_length": np.nan,
        "mean_rg": float(chunk_weighted_rg_df["rg_ca_parent_model_window"].mean()),
        "median_rg": float(chunk_weighted_rg_df["rg_ca_parent_model_window"].median()),
        "p05_rg": float(np.percentile(chunk_weighted_rg_df["rg_ca_parent_model_window"].dropna(), 5)),
        "p95_rg": float(np.percentile(chunk_weighted_rg_df["rg_ca_parent_model_window"].dropna(), 95)),
        "dataset_view": "chunk_weighted_parent_model_window",
    }])
], ignore_index=True)

save_table(chunk_weighted_rg_df, "chunk_weighted_parent_window_radius_of_gyration")
save_table(rg_summary, "radius_of_gyration_summary")
display(rg_summary)

,split,n,median_model_valid_length,mean_rg,median_rg,p05_rg,p95_rg,dataset_view
0,test,1120,137.0,16.517642,15.620952,12.319620,22.985284,parent_model_window
1,train,18024,200.0,17.754377,17.510757,12.089440,24.280851,parent_model_window
2,validation,608,146.0,16.678862,16.178519,11.889951,22.274303,parent_model_window
3,train,27706,NaN,18.605040,18.386293,12.720066,24.843483,chunk_weighted_parent_model_window


In [16]:
fig, ax = plt.subplots(figsize=(7, 5))
for split in ["train", "validation", "test"]:
    sub = parent_rg_df[parent_rg_df["split"] == split]
    ax.scatter(sub["model_valid_length"], sub["rg_ca_parent_model_window"], s=8, alpha=0.35, label=f"{split} parent window")
ax.set_title("Radius of gyration versus model-window length")
ax.set_xlabel("Valid residues in the 256-residue model window")
ax.set_ylabel("Cα radius of gyration (Å)")
ax.legend(markerscale=2)
save_current_figure("parent_window_rg_vs_model_length")
plt.show() if SHOW_FIGURES else plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(parent_rg_df.loc[parent_rg_df["split"] == "train", "rg_ca_parent_model_window"].dropna(), bins=60, alpha=0.6, label="train parent windows")
ax.hist(chunk_weighted_rg_df["rg_ca_parent_model_window"].dropna(), bins=60, alpha=0.45, label="chunk-weighted train parent windows")
ax.set_title("Training compactness: parent-window versus chunk-weighted view")
ax.set_xlabel("Cα radius of gyration (Å)")
ax.set_ylabel("Count")
ax.legend()
save_current_figure("train_rg_parent_vs_chunk_weighted")
plt.show() if SHOW_FIGURES else plt.close(fig)

## 6. Train-only coordinate normalisation

V4/V4i computes coordinate normalisation statistics from the **filtered training parent chains only**, after fixed-window padding/cropping and centring. Validation/test statistics are not used for preprocessing, which avoids leakage. This lightweight audit notebook documents that preprocessing path rather than rerunning a second full coordinate-normalisation pass; the submitted training notebook contains the actual call to `build_normalization_stats_from_dataframe(train_df)`.

In [17]:
print("Train-only coordinate normalisation documentation:")
display(normalisation_summary)

Train-only coordinate normalisation documentation:


,source,training_notebook_helper,centering_helper,statistics_split,validation_or_test_used,note
0,documented_from_v4_v4i_training_notebook,build_normalization_stats_from_dataframe(train...,"centre_coordinates(coords, mask)",filtered training parent chains only,False,The standalone audit avoids a second heavy coo...


## 7. CATH-label composition

The record-level `CATH` field is used here to describe the structural-label composition of the modelling splits. This does not change model training, but it helps show whether the dataset is structurally diverse and whether some topology labels are more common than others.

In [18]:
def topology_label(label: str) -> str:
    parts = str(label).split(".")
    return ".".join(parts[:3]) if len(parts) >= 3 else str(label)

cath_rows = []
for _, row in metadata_df.iterrows():
    labels = [label for label in str(row.get("cath_labels", "")).split(";") if label]
    if not labels:
        cath_rows.append({"record_id": row["record_id"], "split": row["split"], "cath_label": "missing", "cath_topology": "missing"})
    else:
        for label in labels:
            cath_rows.append({"record_id": row["record_id"], "split": row["split"], "cath_label": label, "cath_topology": topology_label(label)})

cath_long_df = pd.DataFrame(cath_rows)
labels_per_structure = metadata_df[["record_id", "split", "n_cath_labels"]].copy()
labels_per_structure_summary = (
    labels_per_structure.groupby("split", as_index=False)
    .agg(
        n_records=("record_id", "count"),
        mean_cath_labels=("n_cath_labels", "mean"),
        median_cath_labels=("n_cath_labels", "median"),
        max_cath_labels=("n_cath_labels", "max"),
        fraction_without_cath_label=("n_cath_labels", lambda s: float((s == 0).mean())),
    )
    .sort_values("split")
)

top_train_topologies = (
    cath_long_df[cath_long_df["split"] == "train"]
    .groupby("cath_topology", as_index=False)
    .agg(count=("record_id", "count"), n_records=("record_id", "nunique"))
    .sort_values("count", ascending=False)
    .head(20)
)

save_table(cath_long_df, "cath_labels_long")
save_table(labels_per_structure, "cath_labels_per_structure")
save_table(labels_per_structure_summary, "cath_labels_per_structure_summary")
save_table(top_train_topologies, "top_train_cath_topologies")

display(labels_per_structure_summary)
print("Top training CATH topology labels:")
display(top_train_topologies)

,split,n_records,mean_cath_labels,median_cath_labels,max_cath_labels,fraction_without_cath_label
0,cath_nodes,1820,1.982418,2.0,5,0.0
1,test,1120,1.081250,1.0,3,0.0
2,train,18024,1.266534,1.0,5,0.0
3,unknown,96,1.197917,1.0,3,0.0
4,validation,608,1.039474,1.0,2,0.0


Top training CATH topology labels:


,cath_topology,count,n_records
732,3.40.50,3541,3541
417,2.60.40,1178,1178
0,1.10.10,867,867
505,3.20.20,827,827
650,3.30.70,817,817
401,2.60.120,653,653
255,1.20.5,352,352
355,2.30.30,351,351
257,1.20.58,312,312
278,1.25.40,310,310


In [19]:
fig, ax = plt.subplots(figsize=(7, 4))
for split in ["train", "validation", "test"]:
    vals = labels_per_structure.loc[labels_per_structure["split"] == split, "n_cath_labels"].astype(int).to_numpy()
    if len(vals):
        max_labels = max(vals.max(), 1)
        bins = np.arange(-0.5, max_labels + 1.5, 1)
        ax.hist(vals, bins=bins, alpha=0.45, label=split)
ax.set_title("CATH labels per structure")
ax.set_xlabel("Number of CATH labels on a record")
ax.set_ylabel("Number of records")
ax.legend()
save_current_figure("cath_labels_per_structure_distribution")
plt.show() if SHOW_FIGURES else plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
plot_df = top_train_topologies.sort_values("count", ascending=True)
ax.barh(plot_df["cath_topology"], plot_df["count"])
ax.set_title("Top 20 CATH topology labels in the training split")
ax.set_xlabel("Label count")
ax.set_ylabel("CATH topology")
save_current_figure("top20_train_cath_topologies")
plt.show() if SHOW_FIGURES else plt.close(fig)

## 8. Split integrity checks

These checks are intentionally simple and conservative. Exact record-ID overlap between train/validation/test would be a direct leakage risk. The separate CATH-topology overlap is not necessarily leakage, but it helps interpret how much structural-class sharing exists between splits.

In [20]:
model_splits = ["train", "validation", "test"]
ids_by_split = {split: set(metadata_df.loc[metadata_df["split"] == split, "record_id"]) for split in model_splits}
leakage_rows = []
for i, split_a in enumerate(model_splits):
    for split_b in model_splits[i + 1:]:
        overlap = sorted(ids_by_split[split_a] & ids_by_split[split_b])
        leakage_rows.append({
            "split_a": split_a,
            "split_b": split_b,
            "exact_record_id_overlap": len(overlap),
            "example_overlaps": ";".join(overlap[:10]),
        })
leakage_df = pd.DataFrame(leakage_rows)

cath_sets = {
    split: set(cath_long_df.loc[(cath_long_df["split"] == split) & (cath_long_df["cath_topology"] != "missing"), "cath_topology"])
    for split in model_splits
}
cath_overlap_rows = []
for i, split_a in enumerate(model_splits):
    for split_b in model_splits[i + 1:]:
        overlap = sorted(cath_sets[split_a] & cath_sets[split_b])
        union = cath_sets[split_a] | cath_sets[split_b]
        cath_overlap_rows.append({
            "split_a": split_a,
            "split_b": split_b,
            "shared_cath_topologies": len(overlap),
            "jaccard_overlap": float(len(overlap) / len(union)) if union else np.nan,
        })
cath_overlap_df = pd.DataFrame(cath_overlap_rows)

save_table(leakage_df, "exact_split_id_overlap")
save_table(cath_overlap_df, "cath_topology_overlap_between_splits")

print("Exact record-ID overlap between modelling splits:")
display(leakage_df)
print("CATH-topology overlap between modelling splits:")
display(cath_overlap_df)

Exact record-ID overlap between modelling splits:


,split_a,split_b,exact_record_id_overlap,example_overlaps
0,train,validation,0,
1,train,test,0,
2,validation,test,0,


CATH-topology overlap between modelling splits:


,split_a,split_b,shared_cath_topologies,jaccard_overlap
0,train,validation,0,0.0
1,train,test,0,0.0
2,validation,test,0,0.0


## 9. Dataset decisions and rationale

This table is designed to be copied into the report or converted into prose. It links each preprocessing decision to the dataset statistic or modelling issue that motivated it.

In [21]:
train_filter = filter_summary.loc[filter_summary["split"] == "train"].iloc[0]
train_lengths = length_summary.loc[length_summary["split"] == "train"].iloc[0]
train_chunks = chunk_summary.iloc[0]
adj_row = geometry_summary.loc[geometry_summary["metric"] == "adjacent_CA"].iloc[0]
rg_train_window = rg_summary.loc[(rg_summary["dataset_view"] == "parent_model_window") & (rg_summary["split"] == "train")].iloc[0]
normalisation = normalisation_summary.iloc[0]

rationale_rows = [
    {
        "decision": "Preserve provided train/validation/test split",
        "dataset_statistic_or_issue": f"The split file assigns {int(filter_summary['total_examples'].sum())} records to modelling splits, with additional auxiliary/non-model records present.",
        "implementation_in_v4_v4i": "Record names are mapped through chain_set_splits.json; train/validation/test take precedence over cath_nodes.",
        "potential_limitation": "The split strategy itself is inherited from the provided dataset rather than redesigned.",
        "report_ready_interpretation": "I preserved the provided split to avoid accidental train/test leakage and treated cath_nodes as metadata rather than a modelling split.",
    },
    {
        "decision": "Filter only invalid or empty examples",
        "dataset_statistic_or_issue": f"In the training split, {int(train_filter['filtered_out'])} of {int(train_filter['total_examples'])} examples were filtered out by the V4-style keep rule.",
        "implementation_in_v4_v4i": "keep_example is true only when a record has at least one valid residue and no NaN/Inf values after fixed-window preprocessing.",
        "potential_limitation": "This is deliberately conservative and does not remove unusual but valid protein geometries.",
        "report_ready_interpretation": "The preprocessing avoids silently training on empty/invalid coordinate tensors without over-cleaning the dataset.",
    },
    {
        "decision": "Use a 256-residue fixed model window with masks",
        "dataset_statistic_or_issue": f"Training parent-chain lengths have median {train_lengths['median']:.1f} residues and {100 * train_lengths['fraction_longer_than_window']:.1f}% exceed {MAX_SEQ_LENGTH} residues.",
        "implementation_in_v4_v4i": "Backbones are padded/cropped to MAX_SEQ_LENGTH and masked so padded residues do not contribute to losses or metrics.",
        "potential_limitation": "A single fixed window cannot represent all long-range context from very long chains.",
        "report_ready_interpretation": "Mask-aware fixed windows make batching simple while retaining the ability to model variable-length proteins.",
    },
    {
        "decision": "Expand long training chains into overlapping chunks",
        "dataset_statistic_or_issue": f"The filtered training set becomes {int(train_chunks['training_chunks'])} windows from {int(train_chunks['train_parent_chains'])} parent chains; the maximum parent contributes {int(train_chunks['max_chunks_per_parent'])} chunks.",
        "implementation_in_v4_v4i": f"Only the training split is expanded into overlapping {MAX_SEQ_LENGTH}-residue windows using stride {TRAIN_CHUNK_STRIDE}.",
        "potential_limitation": "Long parent chains are up-weighted because they contribute more training windows.",
        "report_ready_interpretation": "Chunking reduces information loss from long chains but is tracked as a possible sampling-bias source.",
    },
    {
        "decision": "Centre coordinates and compute train-only normalisation statistics",
        "dataset_statistic_or_issue": f"The V4/V4i notebook computes normalisation statistics from filtered training parent windows only.",
        "implementation_in_v4_v4i": "Each fixed-window backbone is centred by valid backbone atoms; the final training notebook computes coordinate mean/std from filtered training parent windows only using build_normalization_stats_from_dataframe(train_df).",
        "potential_limitation": "Normalisation uses parent windows rather than every overlapping training chunk; this mirrors V4/V4i but could be compared in future work.",
        "report_ready_interpretation": "Train-only normalisation improves optimisation while avoiding validation/test leakage.",
    },
    {
        "decision": "Use geometry-aware diagnostics and losses",
        "dataset_statistic_or_issue": f"Training parent model windows have mean adjacent Cα distance {adj_row['mean']:.2f} Å with std {adj_row['std']:.2f} Å.",
        "implementation_in_v4_v4i": "The model tracks adjacent Cα and backbone bond statistics, and V4 uses geometry-aware loss terms alongside denoising loss.",
        "potential_limitation": "Local geometry constraints alone do not guarantee global fold realism.",
        "report_ready_interpretation": "Backbone geometry statistics from the training data motivate local structural metrics and losses.",
    },
    {
        "decision": "Track radius of gyration / compactness",
        "dataset_statistic_or_issue": f"Training parent model windows have median Cα Rg {rg_train_window['median_rg']:.2f} Å, providing a real-data scale for detecting collapse.",
        "implementation_in_v4_v4i": "Radius of gyration is measured for real structures/chunks and compared with generated samples in diagnostics.",
        "potential_limitation": "Rg is a global compactness summary, not a full measure of protein-likeness.",
        "report_ready_interpretation": "Compactness statistics make the collapse failure mode quantitative rather than purely visual.",
    },
]

rationale_df = pd.DataFrame(rationale_rows)
save_table(rationale_df, "dataset_decisions_and_rationale")
display(rationale_df)

,decision,dataset_statistic_or_issue,implementation_in_v4_v4i,potential_limitation,report_ready_interpretation
0,Preserve provided train/validation/test split,The split file assigns 19752 records to modell...,Record names are mapped through chain_set_spli...,The split strategy itself is inherited from th...,I preserved the provided split to avoid accide...
1,Filter only invalid or empty examples,"In the training split, 0 of 18024 examples wer...",keep_example is true only when a record has at...,This is deliberately conservative and does not...,The preprocessing avoids silently training on ...
2,Use a 256-residue fixed model window with masks,Training parent-chain lengths have median 204....,Backbones are padded/cropped to MAX_SEQ_LENGTH...,A single fixed window cannot represent all lon...,Mask-aware fixed windows make batching simple ...
3,Expand long training chains into overlapping c...,The filtered training set becomes 27706 window...,Only the training split is expanded into overl...,Long parent chains are up-weighted because the...,Chunking reduces information loss from long ch...
4,Centre coordinates and compute train-only norm...,The V4/V4i notebook computes normalisation sta...,Each fixed-window backbone is centred by valid...,Normalisation uses parent windows rather than ...,Train-only normalisation improves optimisation...
5,Use geometry-aware diagnostics and losses,Training parent model windows have mean adjace...,The model tracks adjacent Cα and backbone bond...,Local geometry constraints alone do not guaran...,Backbone geometry statistics from the training...
6,Track radius of gyration / compactness,Training parent model windows have median Cα R...,Radius of gyration is measured for real struct...,"Rg is a global compactness summary, not a full...",Compactness statistics make the collapse failu...


## Report-ready summary

The audit supports the following interpretation:

The dataset was not treated as a black box. Before training, the modelling splits were preserved, coordinate records were checked for validity, variable-length proteins were mapped into fixed-size masked windows, long training chains were expanded into overlapping chunks, and coordinate normalisation statistics were computed only from the training split. The additional analysis quantifies the length distribution, chunking bias, local backbone geometry, compactness/Rg distribution, CATH-label composition, and direct split-overlap checks.

The main remaining limitations are also clear. Overlapping chunks improve coverage of long chains but give long parent chains more influence during training. Random proper 3D rotation augmentation, length-balanced sampling, CATH-balanced sampling, or normalisation computed over the exact chunk distribution could be explored in future work. These were not added to the final model because the priority was a reliable working diffusion prototype with transparent diagnostics.